# 02 — Batch Product Harness

This notebook is the **batch operations gateway** for the Product Evidence Harness. It runs many product rows and makes the concise row packet plus browser-visible verdicts the default review surface.

```mermaid
flowchart TD
    A[Input CSV/XLSX] --> B[Batch runner]
    B --> C[Product evidence run per row]
    C --> V[Browser-visible content verification]
    V --> D[Production-ready rows]
    V --> E[Review queue rows]
    D --> F[final_submission.csv]
    E --> G[review_queue.csv]
    C --> H[Concise row packets + browser_visible artifacts]
```


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

for path in (PROJECT_ROOT, SRC_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT


In [ ]:
from product_evidence_harness import HarnessConfig

config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
tournament = getattr(config, "tournament", None)

pd.Series({
    "output_dir": str(getattr(config, "output_dir", "output")),
    "tournament_enabled": getattr(tournament, "enabled", None),
    "max_serp_credits": getattr(tournament, "max_serp_credits", None),
    "write_review_pack": getattr(config, "write_review_pack", None),
    "browser_visible_verify": getattr(config, "enable_browser_visible_verification", None),
    "require_browser_visible_product_content": getattr(config, "require_browser_visible_product_content", None),
    "browser_visible_capture": getattr(config, "browser_visible_capture_enabled", None),
    "browser_visible_llm": getattr(config, "browser_visible_llm_enabled", None),
    "browser_visible_top_k": getattr(config, "browser_visible_top_k", None),
}).to_frame("value")


## 1. Input file

Required columns: `row_id`, `main_text`, `country_code`. Optional columns: `ean`, `retailer_name`, `language_code`, `region`. Read EAN/GTIN as text.


In [ ]:
input_path = PROJECT_ROOT / "data" / "products.csv"
output_path = PROJECT_ROOT / "outputs" / "final_submission.csv"
workers = 4

print("Input:", input_path)
print("Output:", output_path)
print("Workers:", workers)


In [ ]:
if input_path.exists():
    preview = pd.read_csv(input_path, dtype=str)
    print("Rows:", len(preview))
    display(preview.head())
else:
    print("Input file not found. Update input_path above.")


## 2. Run batch command

The command is shown first. Uncomment the final line to run from the notebook. For long batches, terminal execution is usually easier to monitor.


In [ ]:
cmd = [
    sys.executable, "batch_main.py",
    "--input", str(input_path),
    "--output", str(output_path),
    "--workers", str(workers),
]
print("Command:")
print(" ".join(cmd))

# Uncomment to execute from notebook:
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


## 3. Read final output

`final_submission.csv` is the operational output. Use production and visible-content gates to decide automated handoff.


In [ ]:
if output_path.exists():
    df = pd.read_csv(output_path, dtype=str)
    print("Rows:", len(df))
    display(df.head())
else:
    df = pd.DataFrame()
    print("Run the batch first to create:", output_path)


In [ ]:
handoff_cols = [
    "row_id", "main_text", "country_code", "retailer_name", "ean",
    "product_url", "verified_exact_url", "production_url_ready", "production_url_status",
    "browser_openable", "user_visible_product_match", "user_visible_status", "user_visible_page_type",
    "highly_scrapable", "exact_product_url_match", "needs_review", "confidence",
    "quality_tier", "coding_readiness_status", "failure_taxonomy", "production_url_reasons",
    "product_coding_input_path",
]

if not df.empty:
    display(df[[c for c in handoff_cols if c in df.columns]].head(50))


## 4. Production handoff and review routing

Automated handoff requires `production_url_ready=true`, `user_visible_product_match=true`, `needs_review=false`, and passed champion confirmation. Rows outside this rule are review-only.


In [ ]:
def truthy(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])

if not df.empty:
    ready_mask = pd.Series(False, index=df.index)
    if "production_url_ready" in df.columns and "needs_review" in df.columns:
        ready_mask = truthy(df["production_url_ready"]) & ~truthy(df["needs_review"])
    if "user_visible_product_match" in df.columns:
        ready_mask = ready_mask & truthy(df["user_visible_product_match"])

    print("Production-ready rows by CSV gates:", int(ready_mask.sum()), "/", len(df))
    display(df.loc[ready_mask, [c for c in handoff_cols if c in df.columns]].head(25))

    review_df = df.loc[~ready_mask].copy()
    print("Review rows:", len(review_df))
    display(review_df[[c for c in handoff_cols if c in review_df.columns]].head(25))


## 5. Inspect row packet and browser-visible verdicts

Pick one row and inspect the default row packet. Use visible verdicts when a URL opens but may show a wrong/rerouted page.


In [ ]:
output_dir = Path(getattr(config, "output_dir", PROJECT_ROOT / "output"))
selected_row_id = str(df.iloc[0].get("row_id", "")) if not df.empty else "PUT_ROW_ID_HERE"
row_dir = output_dir / selected_row_id

print("Selected row artifact directory:", row_dir)
expected = ["final_row.csv", "review_summary.md", "review_decision.json", "candidate_decisions.csv", "browser_visible_verdicts.json", "browser_visible", "product_coding_input.json"]

if row_dir.exists():
    existing = {p.name for p in row_dir.iterdir()}
    display(pd.DataFrame({"artifact": expected, "exists": [name in existing for name in expected]}))
else:
    print("Row directory not found. Update selected_row_id or run the batch.")


In [ ]:
visible_path = row_dir / "browser_visible_verdicts.json"
if visible_path.exists():
    visible = json.loads(visible_path.read_text(encoding="utf-8"))
    rows = []
    for url, verdict in visible.items():
        rows.append({
            "url": url,
            "status": verdict.get("status"),
            "page_type": verdict.get("browser_visible_page_type"),
            "user_visible_product_match": verdict.get("user_visible_product_match"),
            "survives_visible_check": verdict.get("champion_should_survive_visible_check"),
            "confidence": verdict.get("user_visible_content_confidence"),
            "rerouted": verdict.get("rerouted_or_not"),
            "screenshot_path": verdict.get("screenshot_path"),
        })
    display(pd.DataFrame(rows))
else:
    print("browser_visible_verdicts.json not found")


In [ ]:
browser_dir = row_dir / "browser_visible"
if browser_dir.exists():
    files = sorted(p for p in browser_dir.iterdir() if p.is_file())
    display(pd.DataFrame({"file": [p.name for p in files], "path": [str(p) for p in files]}).head(50))
else:
    print("browser_visible folder not found")


## 6. Review summary and candidate decisions


In [ ]:
summary_path = row_dir / "review_summary.md"
candidate_path = row_dir / "candidate_decisions.csv"
decision_path = row_dir / "review_decision.json"

if summary_path.exists():
    print(summary_path)
    print("\nFirst 1600 characters:\n")
    print(summary_path.read_text(encoding="utf-8")[:1600])
else:
    print("review_summary.md not found")


In [ ]:
if candidate_path.exists():
    candidates = pd.read_csv(candidate_path, dtype=str)
    review_columns = [
        "rank", "selected", "decision", "url", "confidence", "validation_status",
        "identity_status", "exact_product_check", "country_check", "retailer_check",
        "scrapable", "product_page", "reason",
    ]
    display(candidates[[c for c in review_columns if c in candidates.columns]].head(50))
else:
    print("candidate_decisions.csv not found")


## 7. Batch metrics


In [ ]:
metrics_path = output_path.parent / "metrics.json"

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
else:
    print("metrics.json not found:", metrics_path)
